# RSHazeNet - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains RSHazeNet on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: RSHazeNet  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [ ]:
#@title 1. Install Dependencies
!pip install timm pytorch-msssim lpips scikit-image thop gdown

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/RSHazeNet_Results/weights"

In [ ]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

# Remove corrupted files if they exist
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

In [ ]:
#@title 4. Clone RSHazeNet Repository
!git clone https://github.com/sumire25/RSHazeNet.git
%cd RSHazeNet
!ls

In [ ]:
#@title 5. Download Pretrained Weights
!gdown "1Leyg1sw4x48wEo5zsPKBGtVaCF5NWdY_" -O rs_haze.pth
!ls -lh rs_haze.pth

In [ ]:
#@title 6. Compute FLOPs and Parameters
import torch
from thop import profile
from model import RSHazeNet

model = RSHazeNet().cuda()
model.eval()

dummy_input = torch.randn(1, 3, 512, 512).cuda()
macs, params = profile(model, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- RSHazeNet Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

## Phase 1: Test Pretrained Weights

In [ ]:
#@title 7. Run Inference with Pretrained Weights
!python infer.py \
  --test_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  --test_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/ \
  --weights rs_haze.pth \
  --result_path /content/drive/MyDrive/RSHazeNet_Results/pretrained_results/

In [ ]:
#@title 8. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/RSHazeNet_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

## Phase 2: Train from Scratch

In [ ]:
#@title 9. Train RSHazeNet
!python train.py \
  --train_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/input/ \
  --train_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/train/target/ \
  --val_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/input/ \
  --val_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/valid/target/ \
  --epochs 100 \
  --lr 0.0002 \
  --batch_size_train 8 \
  --batch_size_val 8 \
  --patch_size_train 256 \
  --patch_size_val 256 \
  --val_freq 3 \
  --save_freq 20 \
  --save_dir /content/drive/MyDrive/RSHazeNet_Results/weights/

## Phase 3: Test Trained Weights

In [ ]:
#@title 10. Run Inference with Trained Weights
!python infer.py \
  --test_input /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/input/ \
  --test_target /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/ \
  --weights /content/drive/MyDrive/RSHazeNet_Results/weights/model_best.pth \
  --result_path /content/drive/MyDrive/RSHazeNet_Results/trained_results/

In [ ]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/RSHazeNet_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/